# Unit 2 Assignment: Building a Mixture of Experts (MoE)
##**Topic:** Advanced Architecture using Groq API  
##**Tools:** Python, Groq API, Dotenv

In [2]:
!pip install groq python-dotenv

In [3]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from groq import Groq

# Securely enter API key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

client = Groq(api_key=os.environ["GROQ_API_KEY"])

Enter your Groq API Key: ··········


##Define Model and Experts (MoE Setup)

In [12]:
MODEL_NAME = "llama-3.1-8b-instant"

MODEL_CONFIG = {
    "technical": {
        "system_prompt": """You are a Technical Support Expert.
Be precise, logical, and code-focused.
Provide debugging steps and corrected code."""
    },
    "billing": {
        "system_prompt": """You are a Billing Support Expert.
Be empathetic and policy-driven.
Explain refunds, charges, and subscriptions clearly."""
    },
    "sales": {
        "system_prompt": """You are a Sales Expert.
Be persuasive and informative.
Help customers understand product features and benefits."""
    },
    "general": {
        "system_prompt": """You are a General Support Assistant.
Be friendly and helpful."""
    }
}

In [13]:
def route_prompt(user_input):

    router_instruction = f"""
Classify the following text into one of these categories:
[technical, billing, sales, general]

Return ONLY the category name.

Text:
{user_input}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are an intent classifier."},
            {"role": "user", "content": router_instruction}
        ],
        temperature=0
    )

    category = response.choices[0].message.content.strip().lower()

    return category

In [14]:
def get_bitcoin_price():
    # Mock data (since no real API used)
    return "Current Bitcoin price is $62,450 (mock data)."

In [15]:
def process_request(user_input):

    #  TOOL ROUTING (Bonus)
    if "bitcoin" in user_input.lower():
        return "tool", get_bitcoin_price()

    # Step 1: Route
    category = route_prompt(user_input)

    # Safety fallback
    if category not in MODEL_CONFIG:
        category = "general"

    # Step 2: Load expert prompt
    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    # Step 3: Call expert model
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ],
        temperature=0.7
    )

    final_answer = response.choices[0].message.content

    return category, final_answer

In [16]:
category, answer = process_request("My Python script is throwing an IndexError.")
print("Routed to:", category)
print("Response:\n", answer)

Routed to: technical
Response:
 When you encounter an `IndexError` in Python, it's usually because you're trying to access an element of a list or other sequence that doesn't exist. This can happen when you try to access an index that is higher than the size of the sequence minus one.

To debug this issue, we'll need to see more code. Please provide the relevant part of your script where the error is occurring. 

However, in general, here are some common causes of `IndexError` and how to fix them:

1. **Out of range indexing**:
   - Cause: Trying to access an index that doesn't exist.
   - Fix: Check the size of your list and make sure you're not trying to access an index that's higher than the last valid index.

2. **Negative indexing**:
   - Cause: Trying to access an index that's less than zero.
   - Fix: Make sure you're passing a non-negative index to your list.

Here's an example of how you might debug this issue:

```python
# assuming 'my_list' is the list you're trying to acces

In [17]:
category, answer = process_request("I was charged twice for my subscription.")
print("Routed to:", category)
print("Response:\n", answer)

Routed to: billing
Response:
 I'm so sorry to hear that you've been charged twice for your subscription. I'm here to help you resolve this issue. 

To better understand what happened, can you please provide me with some more information?

1. Can you confirm the date of the first charge and the amount you were charged?
2. Can you also confirm the date of the second charge and the amount you were charged?
3. What type of subscription do you have (monthly, annual, etc.)?
4. Have you made any recent changes to your subscription (e.g., upgraded/downgraded, cancelled, etc.)?

As per our company's policy, we do not intentionally charge customers twice for the same subscription. There might be an error on our end that caused this issue. I'll do my best to investigate and resolve this for you.

Once I have all the necessary information, I can proceed with the following steps:

1. Identify the error and verify it's not a duplicate transaction.
2. Refund the duplicate charge to your original paym

In [18]:
category, answer = process_request("What are the benefits of upgrading to premium?")
print("Routed to:", category)
print("Response:\n", answer)

Routed to: sales
Response:
 You're considering upgrading to our premium product? That's a fantastic decision. Our premium product is designed to take your experience to the next level, offering a range of benefits that will enhance your productivity, convenience, and overall satisfaction.

**Enhanced Features:**

1. **Advanced Security:** Our premium product includes advanced security features, such as two-factor authentication, encryption, and secure data storage, to protect your sensitive information.
2. **Unlimited Storage:** Enjoy unlimited storage space, so you can store as many files, photos, and videos as you want, without worrying about running out of space.
3. **Priority Support:** Get priority access to our dedicated support team, who will be available to assist you 24/7, ensuring that any issues are resolved quickly and efficiently.
4. **Exclusive Content:** Access exclusive content, such as expert tutorials, webinars, and industry insights, to help you stay up-to-date with 

In [19]:
category, answer = process_request("What is the current price of Bitcoin?")
print("Routed to:", category)
print("Response:\n", answer)

Routed to: tool
Response:
 Current Bitcoin price is $62,450 (mock data).
